# Selective pneumonia classification on chest X-rays

The official split contains 4,708 training, 524 validation and 624 test images at 28 × 28 pixels. Three seeds train two three-block CNNs for 12 epochs each. Clean validation Brier score selects the checkpoint. All arms see the same test images and the same fixed corruption. Coverage is calculated separately for each model and test condition.

Published result: At 80% coverage, error on corrupted X-rays fell from 22.5% to 12.4% with augmentation. Clean-image error rose from 8.1% to 11.2%. The corrupted test uses simulated blur, gain and noise on the same X-rays. This measures robustness to that perturbation within one dataset.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'pneumoniamnist-selective-cnn-real-data-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", 'numpy==2.2.6', 'torch==2.14.1', 'scikit-learn==1.7.2'], check=True)

### download.py

In [ ]:
(root / 'download.py').write_text('"""Fetch the official 28px release and check the publisher\'s checksum."""\n\nimport hashlib\nfrom pathlib import Path\nfrom urllib.request import urlopen\n\n\ndef main():\n    target = Path(__file__).resolve().parent / "data" / "pneumoniamnist.npz"\n    target.parent.mkdir(exist_ok=True)\n    if target.exists():\n        payload = target.read_bytes()\n    else:\n        with urlopen(\n            "https://zenodo.org/records/10519652/files/pneumoniamnist.npz?download=1",\n            timeout=90,\n        ) as response:\n            payload = response.read()\n    if hashlib.md5(payload).hexdigest() != "28209eda62fecd6e6a2d98b1501bb15f":\n        raise ValueError("PneumoniaMNIST checksum mismatch")\n    if not target.exists():\n        target.write_bytes(payload)\n    print(target)\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf8")

### train.py

In [ ]:
(root / 'train.py').write_text('"""Selective chest X-ray classification under a paired image corruption."""\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom sklearn.metrics import roc_auc_score, brier_score_loss\nfrom torch import nn\nfrom torch.nn import functional as F\n\nROOT = Path(__file__).resolve().parent\ntorch.set_num_threads(2)\nSEEDS = [17, 29, 43]\n\n\ndef corrupt(x, g):\n    # Global exposure, blur and additive readout noise, identical at test for both arms.\n    gain = 0.55 + 0.60 * torch.rand((len(x), 1, 1, 1), generator=g)\n    noise = torch.randn(x.shape, generator=g) * 0.075\n    return (\n        gain * F.avg_pool2d(F.pad(x, (1, 1, 1, 1), mode="reflect"), 3, 1) + noise\n    ).clamp(0, 1)\n\n\ndef network():\n    layers = []\n    for a, b in [(1, 16), (16, 32), (32, 48)]:\n        layers.extend(\n            [\n                nn.Conv2d(a, b, 3, padding=1, bias=False),\n                nn.BatchNorm2d(b),\n                nn.ReLU(),\n                nn.MaxPool2d(2),\n            ]\n        )\n    return nn.Sequential(\n        *layers,\n        nn.Flatten(),\n        nn.Linear(48 * 3 * 3, 64),\n        nn.ReLU(),\n        nn.Dropout(0.15),\n        nn.Linear(64, 1),\n        nn.Flatten(start_dim=0),\n    )\n\n\ndef predict(model, x):\n    model.eval()\n    with torch.no_grad():\n        return torch.cat([model(part).sigmoid() for part in x.split(128)]).numpy()\n\n\ndef metrics(p, y):\n    error = (p >= 0.5) != y\n    order = np.argsort(-np.abs(p - 0.5), kind="stable")\n    risks = np.cumsum(error[order]) / np.arange(1, len(y) + 1)\n    return {\n        "auroc": roc_auc_score(y, p),\n        "brier": brier_score_loss(y, p),\n        "aurc": risks.mean(),\n        **{f"risk_at_{c}": risks[int(len(y) * c / 100) - 1] for c in (50, 80, 100)},\n    }\n\n\ndef run(data, seed):\n    train, y = data["train"]\n    val, vy = data["val"]\n    test, ty = data["test"]\n    test_shift = corrupt(test, torch.Generator().manual_seed(9001))\n    output = {}\n    for method in ["standard", "augmented"]:\n        torch.manual_seed(seed)\n        model = network()\n        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)\n        best = float("inf")\n        trace = []\n        g = torch.Generator().manual_seed(seed + 1000)\n        for epoch in range(12):\n            model.train()\n            for ids in torch.randperm(len(y)).split(96):\n                x = train[ids].clone()\n                if method == "augmented":\n                    x[::2] = corrupt(x[::2], g)\n                loss = F.binary_cross_entropy_with_logits(model(x), y[ids])\n                optimizer.zero_grad()\n                loss.backward()\n                optimizer.step()\n            score = brier_score_loss(vy, predict(model, val))\n            trace.append(score)\n            if score < best:\n                best = score\n                state = {k: v.detach().clone() for k, v in model.state_dict().items()}\n        model.load_state_dict(state)\n        checkpoint = ROOT / "checkpoints"\n        checkpoint.mkdir(exist_ok=True)\n        torch.save(state, checkpoint / f"{method}-{seed}.pt")\n        output[method] = {\n            "clean": metrics(predict(model, test), ty.numpy()),\n            "shifted": metrics(predict(model, test_shift), ty.numpy()),\n            "validation_brier": trace,\n        }\n        print(seed, method, output[method]["shifted"], flush=True)\n    return output\n\n\nif __name__ == "__main__":\n    cli = argparse.ArgumentParser()\n    cli.add_argument(\n        "--data",\n        type=Path,\n        default=ROOT / "data/pneumoniamnist.npz",\n    )\n    path = cli.parse_args().data\n    with np.load(path) as raw:\n        data = {\n            split: (\n                torch.tensor(raw[split + "_images"][:, None], dtype=torch.float32)\n                / 255,\n                torch.tensor(raw[split + "_labels"].flatten(), dtype=torch.float32),\n            )\n            for split in ("train", "val", "test")\n        }\n    assert data["train"][0].shape[2:] == (28, 28)\n    runs = {s: run(data, s) for s in SEEDS}\n    means = {\n        m: {\n            split: {\n                k: np.mean([r[m][split][k] for r in runs.values()])\n                for k in runs[17][m][split]\n            }\n            for split in ("clean", "shifted")\n        }\n        for m in runs[17]\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps(\n            {\n                "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),\n                "split_sizes": {k: len(v[1]) for k, v in data.items()},\n                "runs": runs,\n                "means": means,\n            },\n            indent=2,\n        )\n    )\n', encoding="utf8")

### Train and evaluate

In [ ]:
subprocess.run([sys.executable, 'download.py'], cwd=root, check=True)
subprocess.run([sys.executable, 'train.py', '--data', 'data/pneumoniamnist.npz'], cwd=root, check=True)

In [ ]:
import json
report = json.loads((root / "results.json").read_text())
print(json.dumps(report.get("means", report.get("subject_macro")), indent=2))